# MP4: Ewaluacja modelu i wpływ biznesowy

**MajsterPlus — Przewidywanie odejść klientów**

W tym mini-projekcie:
1. Zbudujesz cost matrix dla kampanii reaktywacyjnej
2. Obliczysz zysk na rekord przy threshold=0.5
3. Porównasz wyniki ze strategiami baseline (skontaktuj się ze wszystkimi / z nikim)
4. Zoptymalizujesz threshold klasyfikacji pod kątem maksymalizacji zysku
5. Przeprowadzisz analizę lift curve

**Faza CRISP-DM**: Ewaluacja

---

## 0. Konfiguracja i Reprodukowalność

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Sprawdzenie wersji bibliotek
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — oczekiwano 1.4.x lub 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — oczekiwano 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — oczekiwano <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Wczytanie checkpoint z MP3

In [ ]:
import hashlib, pickle
from pathlib import Path

# Wykrywanie środowiska Colab
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/PUM/checkpoints")
except ImportError:
    DATA_DIR = Path("../2. data")
    CHECKPOINT_DIR = Path("../checkpoints")

checkpoint_file = CHECKPOINT_DIR / "mp3_checkpoint.pkl"

if checkpoint_file.exists():
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    print(f"Wczytano checkpoint z MP3: {list(checkpoint.keys())}")
else:
    print(f"Nie znaleziono checkpoint pod ścieżką {checkpoint_file}")
    print("Możesz kontynuować po uruchomieniu najpierw MP3 lub wczytaniu surowych danych.")
    checkpoint = None

In [ ]:
if checkpoint is not None:
    X_train = checkpoint["X_train"]
    X_test = checkpoint["X_test"]
    y_train = checkpoint["y_train"]
    y_test = checkpoint["y_test"]
    feature_names = checkpoint["feature_names"]
    gender_test = checkpoint.get("gender_test")
    lr_model = checkpoint.get("lr_model")
    rf_model = checkpoint.get("rf_model")
    y_prob_lr = checkpoint.get("y_prob_lr")
    y_prob_rf = checkpoint.get("y_prob_rf")
else:
    raise FileNotFoundError(
        "Nie znaleziono checkpoint. Najpierw uruchom rozwiązanie MP3."
    )

print(f"Wczytano modele i predykcje z MP3")
print(f"Rozmiar zbioru testowego: {len(y_test)}")

## 2. Kontekst biznesowy

MajsterPlus rozważa przeprowadzenie ukierunkowanej kampanii reaktywacyjnej:
- **Koszt kampanii na klienta**: 80 PLN (kupon + koszty operacyjne)
- **Wartość kuponu**: 50 PLN
- **Oczekiwany przychód z reaktywacji**: Mediana total_spend klientów odchodzących w zbiorze testowym

Cost matrix:
| | Faktycznie odchodzący (1) | Faktycznie aktywny (0) |
|---|---|---|
| **Przewidziany jako odchodzący (1)** — skontaktowany | Przychód - Koszt (TP) | -Koszt (FP) |
| **Przewidziany jako aktywny (0)** — nieskontaktowany | 0 (FN) | 0 (TN) |

In [ ]:
CAMPAIGN_COST = 80  # PLN na skontaktowanego klienta

# Oblicz oczekiwany przychód z reaktywacji
# Potrzebujemy total_spend z oryginalnych danych dla odchodzących klientów testowych
# Wczytaj surowe dane, aby uzyskać wartości total_spend
import hashlib
cust_raw = pd.read_csv(DATA_DIR / "customers.csv")
cust_raw["total_spend_numeric"] = (
    cust_raw["total_spend"]
    .str.replace("PLN ", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Mediana total_spend odchodzących klientów w zbiorze testowym
lapsed_test_idx = y_test[y_test == 1].index
# Mapowanie do oryginalnych wierszy klientów
lapsed_test_spend = cust_raw.loc[
    cust_raw.index.isin(lapsed_test_idx), "total_spend_numeric"
]
EXPECTED_REVENUE = lapsed_test_spend.median()
print(f"Koszt kampanii na klienta: {CAMPAIGN_COST} PLN")
print(f"Oczekiwany przychód z reaktywacji (mediana wydatków odchodzących): {EXPECTED_REVENUE:.2f} PLN")

## 3. Budowa cost matrix

**Zadanie**: Zdefiniuj funkcję, która oblicza całkowity zysk z kampanii na podstawie prawdziwych etykiet i predykcji.

Zastanów się nad czterema przypadkami: TP, FP, FN, TN — które z nich mają niezerowy wpływ finansowy?

- **True Positive** (odchodzący, skontaktowany): Przychód - Koszt
- **False Positive** (aktywny, skontaktowany): -Koszt
- **False Negative** (odchodzący, nieskontaktowany): 0
- **True Negative** (aktywny, nieskontaktowany): 0

In [ ]:
# TODO: Zdefiniuj funkcję compute_profit(y_true, y_pred, revenue, cost)
# która zwraca (total_profit, tp_count, fp_count, fn_count, tn_count).
#
# Kroki:
# 1. Konwertuj dane wejściowe na tablice numpy
# 2. Policz TP, FP, FN, TN za pomocą masek logicznych
# 3. Oblicz zysk = TP * (revenue - cost) + FP * (-cost)
# 4. Zwróć zysk i wszystkie cztery liczniki

# TWÓJ KOD TUTAJ


### Zastanów się

Zwróć uwagę na asymetrię w cost matrix: True Positive daje zysk `(Przychód - 80)` PLN, a False Positive kosztuje 80 PLN. Zysk netto na TP wynosi tylko ~60 PLN, podczas gdy strata na FP to 80 PLN.

**TODO**: Co ta nierównowaga oznacza dla agresywności modelu w przewidywaniu „odchodzących"? Napisz swoje uzasadnienie w nowej komórce markdown poniżej (lub jako komentarz).

In [ ]:
# TODO: Napisz swoje uzasadnienie jako komentarz lub instrukcję print.
# Rozważ: Jeśli kara za FP (80 PLN) przewyższa zysk netto z TP (~60 PLN),
# czy model powinien być agresywny (przewidywać wiele pozytywnych) czy konserwatywny
# (przewidywać mniej, ale bardziej pewnych pozytywnych)?

# TWOJA ODPOWIEDŹ TUTAJ


## 4. Zysk przy threshold = 0.5

**Zadanie**: Oblicz zysk dla obu modeli (LogReg, RF) przy domyślnym
threshold 0.5. Oblicz również zysk na rekord.

In [ ]:
# TODO: Przekonwertuj predykcje prawdopodobieństw na predykcje binarne przy threshold 0.5
# dla LogisticRegression i RandomForest.
# Następnie użyj funkcji compute_profit, aby obliczyć zysk dla każdego modelu.
# Wyświetl: liczbę TP, FP, FN, TN, całkowity zysk i zysk na rekord.

# TWÓJ KOD TUTAJ


## 5. Porównanie z baseline

**Zadanie**: Oblicz zysk dla dwóch naiwnych strategii:
1. **Skontaktuj się ze wszystkimi**: Przewiduj wszystkich klientów jako odchodzących
2. **Nie kontaktuj się z nikim**: Przewiduj wszystkich klientów jako aktywnych

In [ ]:
# TODO: Utwórz dwie tablice:
# - y_all_ones: wszystkie predykcje = 1 (skontaktuj się ze wszystkimi)
# - y_all_zeros: wszystkie predykcje = 0 (nie kontaktuj się z nikim)
# Oblicz zysk dla każdej strategii baseline.

# TWÓJ KOD TUTAJ


### Zastanów się

Strategia „skontaktuj się ze wszystkimi" generuje znaczne straty.

**TODO**: Dlaczego kontaktowanie się ze wszystkimi powoduje tak duże straty? Pomyśl o proporcji klas — około 80% klientów jest aktywnych, więc około 80% kontaktów to False Positives po -80 PLN każdy. Oblicz przybliżoną stratę z FP i porównaj ją z zyskiem z TP.

In [ ]:
# TODO: Pokaż obliczenia jawnie:
# - Ile kontaktów to FP? Jaka jest łączna strata z FP?
# - Ile kontaktów to TP? Jaki jest łączny zysk z TP?
# - Dlaczego strata z FP przewyższa zysk z TP?

# TWÓJ KOD TUTAJ


## 6. Optymalizacja threshold

**Zadanie**: Przejdź przez threshold od 0.05 do 0.95 (krok 0.05) i oblicz
zysk przy każdym threshold dla obu modeli. Narysuj wykres zysku vs. threshold.

In [ ]:
# TODO: Przeszukaj threshold za pomocą np.arange(0.05, 1.0, 0.05).
# Dla każdego threshold przekonwertuj prawdopodobieństwa na predykcje binarne
# i oblicz zysk dla LR i RF.
# Zapisz wyniki w listach, a następnie narysuj obie krzywe na jednym wykresie.
# Dodaj linię poziomą dla baseline „skontaktuj się ze wszystkimi".

# TWÓJ KOD TUTAJ


## 7. Analiza optymalnego threshold

**Zadanie**: Zidentyfikuj optymalny threshold dla każdego modelu i przeanalizuj predykcje przy tym threshold.

In [ ]:
# TODO: Znajdź threshold maksymalizujący zysk dla każdego modelu.
# Wyświetl: optymalny threshold, maksymalny zysk, liczbę skontaktowanych klientów,
# liczbę TP, FP i zysk na rekord przy optymalnym threshold.

# TWÓJ KOD TUTAJ


### Zastanów się

Optymalny threshold prawdopodobnie jest poniżej 0.5. Dlaczego? Niższy threshold oznacza przewidywanie WIĘKSZEJ liczby klientów jako odchodzących (wyższy recall, niższy precision). W którym momencie dodatkowy koszt FP przewyższa zysk z TP?

**TODO**: Przy optymalnym threshold oblicz:
- (a) Zysk z TP = liczba TP × (przychód - koszt)
- (b) Strata z FP = liczba FP × koszt
- (c) Zysk netto = (a) - (b)

Czy margines jest wąski? Co by się stało, gdyby koszt kampanii wzrósł o zaledwie 10 PLN?

In [ ]:
# TODO: Rozłóż zysk przy optymalnym threshold na
# składowe: zysk z TP i strata z FP. Omów margines.

# TWÓJ KOD TUTAJ


## 8. Analiza lift curve

**Zadanie**: Utwórz krzywą skumulowanych zysków (lift curve) pokazującą, jaki
odsetek odchodzących klientów wyłapiemy, kontaktując się z górnym X% klientów
(uszeregowanych według przewidywanego prawdopodobieństwa).

In [ ]:
# TODO: Dla modelu RandomForest:
# 1. Posortuj klientów testowych według przewidywanego prawdopodobieństwa (malejąco)
# 2. Oblicz skumulowaną liczbę wyłapanych odchodzących klientów
# 3. Narysuj wykres: % skontaktowanych klientów (oś X) vs % wyłapanych odchodzących (oś Y)
# 4. Dodaj linię przekątną reprezentującą baseline „brak modelu" (losowy)
# 5. Wyświetl % wyłapanych odchodzących przy poziomach kontaktu 10%, 20%, 30% i 50%

# TWÓJ KOD TUTAJ


### Zastanów się

**TODO**: Przy poziomie kontaktu 20% wyłapujesz pewien odsetek odchodzących klientów. Czy to wystarczająco dużo? Jaki jest biznesowy koszt pominięcia pozostałych odchodzących klientów, z którymi się nie skontaktowano?

Rozważ kompromis: kontaktowanie się z większą liczbą klientów zwiększa koszt kampanii (więcej FP), ale wyłapuje więcej odchodzących klientów (więcej TP). Gdzie jest optymalny punkt równowagi?

In [ ]:
# TODO: Omów kompromis między pokryciem a kosztem.
# Jaką część odchodzących klientów jesteśmy gotowi pominąć,
# aby kampania pozostała rentowna?

# TWOJA ODPOWIEDŹ TUTAJ


## 9. Szacowanie rocznego zysku

**Zadanie**: Ekstrapoluj zysk ze zbioru testowego na całą bazę klientów, aby
oszacować roczny wpływ kampanii.

In [ ]:
# TODO: Zbiór testowy stanowi ~20% całej bazy klientów (5000 klientów
# przed usunięciem wartości odstających). Oblicz test_fraction = len(y_test) / 5000.
# Przeskaluj optymalny zysk, aby oszacować roczną wartość kampanii.
# Porównaj również: ile kosztowałaby rocznie strategia „skontaktuj się ze wszystkimi"?

# TWÓJ KOD TUTAJ


## 10. Zapis checkpoint dla MP5

In [ ]:
import pickle

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_mp4 = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_names": feature_names,
    "gender_test": gender_test,
    "lr_model": lr_model,
    "rf_model": rf_model,
    "y_prob_lr": y_prob_lr,
    "y_prob_rf": y_prob_rf,
    "CAMPAIGN_COST": CAMPAIGN_COST,
    "EXPECTED_REVENUE": EXPECTED_REVENUE,
    # Dodaj swoje wyniki:
    # "optimal_threshold_rf": optimal_threshold_rf,
    # "optimal_profit_rf": optimal_profit_rf,
}

with open(CHECKPOINT_DIR / "mp4_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint_mp4, f)
print(f"Checkpoint zapisany: {CHECKPOINT_DIR / 'mp4_checkpoint.pkl'}")

---
*Koniec MP4 — Przejdź do MP5: Porównanie modeli i rekomendacja końcowa*